In [1]:
import os

In [2]:
%pwd

'c:\\Users\\mebra\\Documents\\Coding\\AI_ML_Projects\\AemroVision-Brain-Tumor-Classifier\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\mebra\\Documents\\Coding\\AI_ML_Projects\\AemroVision-Brain-Tumor-Classifier'

**1. Update `config.yaml` file**

Before data ingestion step, this should be there as it is common for all stages.

```yaml
artifacts_root: artifacts
```
For data ingestion:
```yaml
data_ingestion:
  root_dir: artifacts/data_ingestion
  source_URL: https://drive.google.com/file/d/1E_MXAfeGD3Mj7vRlDVgMTKxc_XSj03ya/view?usp=sharing
  local_data_file: artifacts/data_ingestion/brain-tumor-mri-dataset.zip
  unzip_dir: artifacts/data_ingestion
```

**2. Update `params.yaml` file**

Currently no params. So there I need to put placeholder as it shouldn't be empty.
```yaml
key: value
```


**3. Update Entity**

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True) # frozen = True means it is immutable
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

**constants**

Inside `constants/__init__.py` file.

```py
from pathlib import Path

CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
```

**4. Update the configuration**

`configuration.py`

In [ ]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root], verbose=True)

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        
        return DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

**5. Update components**

`conponents/__init__.py`

In [ ]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
    
    def download_file(self) -> str:
        """Fetch data from the url"""
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file: {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?export=download&id="
            gdown.download(prefix+file_id, zip_download_dir)
            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")
        except Exception as e:
            raise e
        
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

**6. Update pipeline**
- Excution is inside pipeline

In [ ]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

**7. Update `main.py`**

In [ ]:
STAGE_NAME = "Data Ingestion Stage"
if __name__ == "__main__":
    try:
        logger.info(f">>>>>>>>>>>>>>> Stage \'{STAGE_NAME}\' is started. <<<<<<<<<<<<<<<")
        stage1 = DataIngestionPipeline()
        stage1.main()
        logger.info(f">>>>>>>>>>>>>>> Stage \'{STAGE_NAME}\'is completed. <<<<<<<<<<<<<<<\n\n x=================x")
    except Exception as e:
        logger.exception(e)
        raise e